# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

**Recommended:** create a venv, then `pip install -r requirements.txt` from the repo root, and register/select a Jupyter kernel for that venv (see `context/ENVIRONMENT_SETUP.md`).

Optional: the cell below uses `uv` and shell paths that target **Linux/macOS** (and `wget`). On **Windows**, prefer `requirements.txt` + the correct kernel instead of running the install cell.

> **After first install, restart the kernel** so packages (especially `torch`, `vllm` if installed) are picked up.

### Optional install cell (Linux/macOS)

Uncomment commands in the next cell if you use `uv`; otherwise rely on `pip install -r requirements.txt`.

In [1]:
# Install deps into the *same* Python as this notebook (avoids wrong paths on Windows / wrong cwd).
# Optional `uv` lines at bottom: Linux/macOS only. On Windows use sys.executable or .venv\Scripts\python.exe.
from pathlib import Path
import sys


def _repo_root() -> Path:
    p = Path.cwd().resolve()
    for d in (p, *p.parents):
        if (d / "judger.py").is_file():
            return d
    return p


ROOT = _repo_root()
REQ = ROOT / "requirements.txt"

print("Repo root:", ROOT)
print("requirements.txt:", REQ, "| exists:", REQ.is_file())
print("This kernel's Python:", sys.executable)
print()

if REQ.is_file():
    print("Install (copy into a terminal, or set RUN_PIP_INSTALL_HERE = True below):")
    print(f'  "{sys.executable}" -m pip install -U pip')
    print(f'  "{sys.executable}" -m pip install -r "{REQ}"')
    print()
    print("Register kernel (after venv is active / using the Python you want):")
    print(f'  "{sys.executable}" -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"')
else:
    print("Could not find requirements.txt next to judger.py — check repo layout.")

# One-shot install into *this* Jupyter kernel (same as running the pip lines above)
RUN_PIP_INSTALL_HERE = False
if RUN_PIP_INSTALL_HERE and REQ.is_file():
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQ)])
    print("Done. Restart the kernel, then continue.")

# Linux/macOS only (optional):
# !wget -qO- https://astral.sh/uv/install.sh | sh
# !uv venv .venv --seed
# !uv pip install -r requirements.txt

Repo root: /mnt/c/Users/sardo/OneDrive/Desktop/Classes/projects/math-qa-llm
requirements.txt: /mnt/c/Users/sardo/OneDrive/Desktop/Classes/projects/math-qa-llm/requirements.txt | exists: True
This kernel's Python: /home/sardo/cse151b-venv/bin/python

Install (copy into a terminal, or set RUN_PIP_INSTALL_HERE = True below):
  "/home/sardo/cse151b-venv/bin/python" -m pip install -U pip
  "/home/sardo/cse151b-venv/bin/python" -m pip install -r "/mnt/c/Users/sardo/OneDrive/Desktop/Classes/projects/math-qa-llm/requirements.txt"

Register kernel (after venv is active / using the Python you want):
  "/home/sardo/cse151b-venv/bin/python" -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"


### Kernel note

Shell activation (`source .venv/...`) does **not** change which Python Jupyter uses. Pick the project venv **kernel** instead.

In [2]:
# No-op: Jupyter runs the interpreter of the *selected kernel*, not the shell you activate here.
print("Using:", __import__("sys").executable)

Using: /home/sardo/cse151b-venv/bin/python


## 2. Imports & Configuration

Paths are resolved from the repo root (folder containing `judger.py`), so the notebook works whether your **current working directory** is the repo root or `notebooks/`.

- `DATA_PATH` — default `data/raw/public.jsonl` (labeled)
- `PRIVATE_PATH` — `data/raw/private.jsonl` (switch `DATA_PATH` to this for submission-only runs; no local accuracy)
- `OUTPUT_PATH` — default under `artifacts/logs/runs/`
- `N_QUESTIONS` — `None` = full file; integer = smoke-test subset (must match generation + scoring)
- `TEST_RANDOM_SUBSET` — if `True` and `N_QUESTIONS` is an int, draw a **random** sample (see `RANDOM_SEED`) instead of the **first** N rows — better for quick sanity checks across the file
- `GPU_ID` — passed to `CUDA_VISIBLE_DEVICES` (single GPU is usually `"0"`)
- `MAX_TOKENS` — max **new** tokens per answer; Qwen3-Thinking needs headroom for long `<think>` chains before `\boxed{}` (default **8192**; **16384** safer on hard items, slower)

In [ ]:
import json
import os
import random
import re
import sys
from pathlib import Path
from typing import Optional


def repo_root() -> Path:
    """Directory that contains `judger.py` (works from repo root or `notebooks/`)."""
    p = Path.cwd().resolve()
    for d in (p, *p.parents):
        if (d / "judger.py").is_file():
            return d
    return p


REPO_ROOT = repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Configuration
# Competition rule: final inference must use this exact model, no alternatives.
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID = "0"  # single-GPU machines usually "0"

PUBLIC_PATH = REPO_ROOT / "data" / "raw" / "public.jsonl"
PRIVATE_PATH = REPO_ROOT / "data" / "raw" / "private.jsonl"

# "public" = local scoring possible; "private" = leaderboard CSV, skip scoring.
DATA_MODE = "public"
DATA_PATH = PUBLIC_PATH if DATA_MODE == "public" else PRIVATE_PATH

# For validation use 10 or 50. For private submission use None (all private rows).
N_QUESTIONS = 50
TEST_RANDOM_SUBSET = True
RANDOM_SEED = 42

# Run naming / artifacts
RUN_NAME = f"adaptive_{DATA_MODE}_v1"
OUTPUT_PATH = REPO_ROOT / "artifacts" / "logs" / "runs" / f"{RUN_NAME}_results.jsonl"
CHECKPOINT_PATH = REPO_ROOT / "artifacts" / "logs" / "runs" / f"{RUN_NAME}_checkpoint.jsonl"
CHECKPOINT_INTERVAL = 50

# Adaptive generation config
USE_ADAPTIVE_GENERATION = True

# Qwen3 thinking budget is a soft prompt/template hint. If unsupported by the
# installed tokenizer, the notebook falls back to a plain prompt plus max_tokens.
PHASE1_THINKING_BUDGET = 1024
PHASE2_THINKING_BUDGET = 4096
PHASE3_THINKING_BUDGET = None

PHASE1_MAX_TOKENS = 2048
PHASE2_MAX_TOKENS = 6144
PHASE3_MAX_TOKENS = 8192

PHASE1_N_SAMPLES = 1
PHASE2_N_SAMPLES = 4
PHASE3_N_SAMPLES = 8

# Legacy/simple generation toggles. Keep off for current adaptive plan.
USE_SELF_CONSISTENCY = False
N_SAMPLES = 4

MAX_TOKENS = PHASE3_MAX_TOKENS

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

from transformers import AutoTokenizer

try:
    from vllm import LLM, SamplingParams

    VLLM_AVAILABLE = True
except ImportError:
    LLM = None  # type: ignore[misc, assignment]
    SamplingParams = None  # type: ignore[misc, assignment]
    VLLM_AVAILABLE = False

from tqdm.auto import tqdm

USE_VLLM = True

print("REPO_ROOT:", REPO_ROOT)
print("DATA_MODE:", DATA_MODE)
print("DATA_PATH:", DATA_PATH, "| exists:", DATA_PATH.is_file())
print("PRIVATE_PATH:", PRIVATE_PATH, "| exists:", PRIVATE_PATH.is_file())
print("N_QUESTIONS:", N_QUESTIONS)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("vLLM available:", VLLM_AVAILABLE, "| USE_VLLM:", USE_VLLM)


## 3. Load the Dataset

Data is **JSONL**: one JSON object per line. Paths in this notebook default to **`data/raw/public.jsonl`** (development) and **`data/raw/private.jsonl`** (submission); only the public file includes labels.

### Fields (by split)

| Field | Public (`public.jsonl`) | Private (`private.jsonl`) |
|------|-------------------------|---------------------------|
| `id` | Integer, unique per problem | Same |
| `question` | Problem text, **LaTeX**; free-form items use **`[ANS]`** placeholders where answers go | Same |
| `options` | Present for **MCQ** (list of LaTeX choices); omitted for **free-form** | Same rule |
| `answer` | **Present**: MCQ → single capital letter string (e.g. `"C"`); free-form → **list of strings** (one entry per `[ANS]`, in order) | **Omitted** (withheld for evaluation) |

### Formats

- **Free-form:** One or more answers; every sub-answer must match for the problem to count as correct when scoring locally.
- **MCQ:** Model should pick one letter matching the labeled option.

Use **`DATA_PATH`** in the config cell to point at public vs private; run scoring (§7) only when every loaded row has an **`answer`** field.

In [ ]:
with open(DATA_PATH, encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

n_mcq = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

if N_QUESTIONS is None:
    data_run = data
    print(f"Inference slice `data_run`: all {len(data_run)} question(s).")
else:
    k = min(int(N_QUESTIONS), len(data))
    if TEST_RANDOM_SUBSET:
        rng = random.Random(RANDOM_SEED)
        data_run = rng.sample(data, k=k)
        print(
            f"Inference slice `data_run`: random sample of {k} question(s), seed={RANDOM_SEED}."
        )
    else:
        data_run = data[:k]
        print(f"Inference slice `data_run`: first {k} question(s).")

_nrun_mcq = sum(bool(d.get("options")) for d in data_run)
print(f"  -> in this slice: {_nrun_mcq} MCQ, {len(data_run) - _nrun_mcq} free-form")

# Preview one MCQ and one free-form item (from full split)
mcq_sample = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
# Phase 2 prompt engineering: Formula-One + Plan-and-Solve+ + EmotionPrompt.
# PHASE 2 PROMPT ENGINEERING - Formula-One + Plan-and-Solve+ + EmotionPrompt
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician with deep knowledge of all areas of mathematics, "
    "from algebra and calculus to number theory and combinatorics. "
    "This problem is very important to my career - please think carefully and be precise.\n\n"
    "Solve using this structured approach:\n"
    "1. UNDERSTAND: Identify what is given and what you need to find.\n"
    "2. PLAN: Write down the key equations, formulas, or theorems you will use.\n"
    "3. SOLVE: Work through each step carefully. Compute intermediate results explicitly. "
    "Pay special attention to arithmetic - do not skip steps.\n"
    "4. VERIFY: Check that your answer satisfies all conditions in the problem. "
    "Check units, sign, and order of magnitude.\n"
    "5. ANSWER: Put your final answer in \\boxed{}.\n\n"
    "Additional rules:\n"
    "- If the problem has multiple blanks ([ANS] placeholders), put ALL answers "
    "comma-separated in ONE \\boxed{} in the order they appear. "
    "Example: \\boxed{3, -7, 42}.\n"
    "- Simplify all fractions and radical expressions completely.\n"
    "- You'd better be sure of your answer."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician with deep knowledge of all areas of mathematics. "
    "This problem is very important to my career - please think carefully and be precise.\n\n"
    "Solve using this structured approach:\n"
    "1. UNDERSTAND: Read the problem and all answer choices carefully.\n"
    "2. PLAN: Identify the relevant concepts, formulas, or theorems that apply.\n"
    "3. SOLVE: Work through the problem step by step. Compute intermediate results "
    "explicitly - do not skip arithmetic steps.\n"
    "4. ELIMINATE: Cross out answer choices that are clearly wrong.\n"
    "5. VERIFY: Confirm your chosen answer is consistent with every condition in the problem.\n"
    "6. ANSWER: On the very last line of your response, write ONLY \\boxed{X} "
    "where X is the letter of the correct answer (A-J). "
    "Do not write any text after \\boxed{}.\n\n"
    "You'd better be sure of your answer."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"

    n_ans = question.count("[ANS]")
    if n_ans > 1:
        hint = (
            f"\n\n[Note: This problem has {n_ans} answers. "
            f"Put all {n_ans} answers comma-separated in ONE \\boxed{{}} "
            f"in the order they appear in the question.]"
        )
        return SYSTEM_PROMPT_MATH, question + hint

    return SYSTEM_PROMPT_MATH, question


# Verify with samples - now prints SYSTEM prompt preview too, so changes are visible.
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"-- {label} SYSTEM prompt preview --")
    print(sys_p[:500], "...\n")
    print(f"-- {label} USER prompt preview --")
    print(usr_p[:200], "...\n")

-- MCQ SYSTEM prompt preview --
You are an expert mathematician with deep knowledge of all areas of mathematics. This problem is very important to my career - please think carefully and be precise.

Solve using this structured approach:
1. UNDERSTAND: Read the problem and all answer choices carefully.
2. PLAN: Identify the relevant concepts, formulas, or theorems that apply.
3. SOLVE: Work through the problem step by step. Compute intermediate results explicitly - do not skip arithmetic steps.
4. ELIMINATE: Cross out answer ch ...

-- MCQ USER prompt preview --
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

-- Free-form SYSTEM prompt preview --
You are an expert mathematician with deep knowledge of all areas of mathematics, from algebra and calculus to number theory and combinatorics. This problem is very important to my career -

## 5. Load Model with vLLM (for general case, vLLM is faster)

Model id comes from **`MODEL_ID`** in the config cell (default **Qwen3-8B-Thinking**). We use **BitsAndBytes INT8** in vLLM (`load_format="bitsandbytes"`), which roughly halves GPU memory versus BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel; keep it >= `N_SAMPLES` for self-consistency

In [ ]:
# Adaptive generation helpers
from collections import Counter
from math import ceil


def extract_boxed(text: str) -> str:
    """Extract last \\boxed{...} content from a response (nested braces supported)."""
    matches = []
    needle = r"\boxed{"
    i = 0
    while i < len(text):
        idx = text.find(needle, i)
        if idx == -1:
            break
        j = idx + len(needle)
        depth = 1
        start = j
        while j < len(text) and depth:
            if text[j] == "{":
                depth += 1
            elif text[j] == "}":
                depth -= 1
                if depth == 0:
                    matches.append(text[start:j])
                    break
            j += 1
        i = idx + 1
    return matches[-1].strip() if matches else ""


def majority_vote(answers: list[str]) -> str:
    """Return the most common answer string from a list."""
    if not answers:
        return ""
    return Counter(answers).most_common(1)[0][0]


def strip_thinking(text: str) -> str:
    """Remove explicit <think>...</think> blocks when present."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def is_uncertain(response: str, finish_reason: str = "") -> bool:
    """Return True if this response should be retried with more compute."""
    if "length" in str(finish_reason).lower():
        return True
    if not extract_boxed(response):
        return True
    answer_only = strip_thinking(response)
    if len(answer_only) < 30:
        return True
    return False


def choose_best_sample(samples: list[str], finish_reasons: list[str]) -> dict:
    """Choose the sample whose boxed answer wins majority; tie-break by longer trace."""
    extracted = [extract_boxed(s) for s in samples]
    nonempty = [e for e in extracted if e]

    if nonempty:
        counts = Counter(nonempty)
        top_count = counts.most_common(1)[0][1]
        tied_answers = {ans for ans, count in counts.items() if count == top_count}
        candidate_indexes = [i for i, ans in enumerate(extracted) if ans in tied_answers]
        best_idx = max(candidate_indexes, key=lambda i: len(samples[i]))
        best_answer = extracted[best_idx]
    else:
        top_count = 0
        best_idx = 0
        best_answer = ""

    threshold = ceil(len(samples) / 2)
    uncertain = is_uncertain(samples[best_idx], finish_reasons[best_idx]) or top_count < threshold
    return {
        "response": samples[best_idx],
        "answer": best_answer,
        "finish_reason": finish_reasons[best_idx],
        "consensus_count": top_count,
        "n_samples": len(samples),
        "uncertain": uncertain,
    }


def make_sampling_params(max_tokens: int, temperature: float = 0.6):
    return SamplingParams(
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=0.95,
        top_k=20,
        min_p=0.0,
        presence_penalty=0.0,
        repetition_penalty=1.0,
    )


def build_chat_prompt(item: dict, thinking_budget: int | None = None, prefix: str = "") -> str:
    """Render a chat prompt; fall back if tokenizer lacks thinking_budget support."""
    system, user = build_prompt(item["question"], item.get("options"))
    if prefix:
        user = prefix + user

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    kwargs = dict(tokenize=False, add_generation_prompt=True, enable_thinking=True)
    if thinking_budget is not None:
        kwargs["thinking_budget"] = thinking_budget

    try:
        return tokenizer.apply_chat_template(messages, **kwargs)
    except TypeError as exc:
        if thinking_budget is None:
            raise
        budget_hint = (
            f"Use at most about {thinking_budget} thinking tokens. "
            "Be concise but do not skip necessary arithmetic.\n\n"
        )
        messages[1]["content"] = budget_hint + user
        kwargs.pop("thinking_budget", None)
        print(f"thinking_budget unsupported by tokenizer; using prompt hint fallback ({exc}).")
        return tokenizer.apply_chat_template(messages, **kwargs)


def load_checkpoint(path: Path) -> dict[str, dict]:
    if not path.exists():
        return {}
    records = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            records[str(rec["id"])] = rec
    print(f"Loaded checkpoint: {len(records)} record(s) from {path}")
    return records


def write_checkpoint(path: Path, records: dict[str, dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for rec in sorted(records.values(), key=lambda r: int(r["id"])):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    gpu_memory_utilization=0.78,  # 8 GB VRAM headroom; lower if OOM
    max_model_len=8192,
    trust_remote_code=True,
    max_num_seqs=8,  # supports optional phase 3 N=8 sampling
    max_num_batched_tokens=8192,
)

sampling_params = make_sampling_params(MAX_TOKENS, temperature=0.6)

print("Model loaded.")


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [8]:
# --- Transformers load path (disabled when using vLLM — do not run both; GPU OOM) ---
# import torch
# from transformers import AutoModelForCausalLM, BitsAndBytesConfig
#
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "left"  # required for correct batched generation in decoder-only LMs
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )
#
# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )
#
# _device = next(llm.parameters()).device
# print("Model device:", _device)


## 6. Generate Responses - Adaptive Multi-Pass

The adaptive path concentrates compute where it is useful:

1. **Phase 1:** one fast response for every question (`thinking_budget=1024`, `max_tokens=2048`).
2. **Phase 2:** retry uncertain questions with more budget and `N=4` majority vote.
3. **Phase 3:** retry the still-uncertain tail with maximum budget and `N=8` majority vote.

The cell writes `CHECKPOINT_PATH` after each phase/batch so interrupted runs can resume. It also assembles `responses` in `data_run` order, so scoring and saving continue to work.


### Generate with vLLM

In [ ]:
if not USE_ADAPTIVE_GENERATION:
    prompts = [build_chat_prompt(item, thinking_budget=None) for item in data_run]
    outputs = llm.generate(prompts, sampling_params)
    response_records = {}
    for item, out in zip(data_run, outputs):
        response = out.outputs[0].text.strip()
        finish_reason = str(out.outputs[0].finish_reason)
        response_records[str(item["id"])] = {
            "id": item["id"],
            "phase_used": 0,
            "response": response,
            "answer": extract_boxed(response),
            "finish_reason": finish_reason,
            "uncertain": is_uncertain(response, finish_reason),
            "n_samples": 1,
            "consensus_count": 1,
        }
    write_checkpoint(CHECKPOINT_PATH, response_records)
else:
    response_records = load_checkpoint(CHECKPOINT_PATH)

    phase1_params = make_sampling_params(PHASE1_MAX_TOKENS, temperature=0.6)
    missing_phase1 = [item for item in data_run if str(item["id"]) not in response_records]
    print(f"Phase 1: {len(missing_phase1)} missing / {len(data_run)} total question(s).")

    for start in tqdm(range(0, len(missing_phase1), CHECKPOINT_INTERVAL), desc="Phase 1", unit="batch"):
        batch = missing_phase1[start : start + CHECKPOINT_INTERVAL]
        prompts = [build_chat_prompt(item, thinking_budget=PHASE1_THINKING_BUDGET) for item in batch]
        outputs = llm.generate(prompts, phase1_params)

        for item, out in zip(batch, outputs):
            response = out.outputs[0].text.strip()
            finish_reason = str(out.outputs[0].finish_reason)
            response_records[str(item["id"])] = {
                "id": item["id"],
                "phase_used": 1,
                "response": response,
                "answer": extract_boxed(response),
                "finish_reason": finish_reason,
                "uncertain": is_uncertain(response, finish_reason),
                "n_samples": 1,
                "consensus_count": 1,
            }
        write_checkpoint(CHECKPOINT_PATH, response_records)

    phase1_uncertain = [
        item for item in data_run
        if response_records[str(item["id"])].get("uncertain")
        and int(response_records[str(item["id"])].get("phase_used", 0)) < 2
    ]
    print(f"Phase 1 complete. {len(phase1_uncertain)} question(s) uncertain.")

    phase2_params = make_sampling_params(PHASE2_MAX_TOKENS, temperature=0.6)
    for item in tqdm(phase1_uncertain, desc="Phase 2", unit="q"):
        prefix = "Previous attempt was unclear. Solve this again carefully from scratch:\n\n"
        if item.get("options"):
            prefix += (
                "After solving, verify your answer by checking the options. "
                "Eliminate any option that fails all conditions.\n\n"
            )
        prompt_text = build_chat_prompt(item, thinking_budget=PHASE2_THINKING_BUDGET, prefix=prefix)
        outputs = llm.generate([prompt_text] * PHASE2_N_SAMPLES, phase2_params)
        samples = [out.outputs[0].text.strip() for out in outputs]
        finish_reasons = [str(out.outputs[0].finish_reason) for out in outputs]
        chosen = choose_best_sample(samples, finish_reasons)
        chosen.update({"id": item["id"], "phase_used": 2})
        response_records[str(item["id"])] = chosen
        write_checkpoint(CHECKPOINT_PATH, response_records)

    phase2_uncertain = [
        item for item in data_run
        if response_records[str(item["id"])].get("uncertain")
        and int(response_records[str(item["id"])].get("phase_used", 0)) < 3
    ]
    print(f"Phase 2 complete. {len(phase2_uncertain)} question(s) still uncertain.")

    phase3_params = make_sampling_params(PHASE3_MAX_TOKENS, temperature=0.7)
    phase3_prefix = (
        "This is a genuinely difficult problem. Before attempting to solve it, "
        "identify the mathematical domain and the key insight needed. Then solve carefully.\n\n"
    )
    for item in tqdm(phase2_uncertain, desc="Phase 3", unit="q"):
        prompt_text = build_chat_prompt(item, thinking_budget=PHASE3_THINKING_BUDGET, prefix=phase3_prefix)
        outputs = llm.generate([prompt_text] * PHASE3_N_SAMPLES, phase3_params)
        samples = [out.outputs[0].text.strip() for out in outputs]
        finish_reasons = [str(out.outputs[0].finish_reason) for out in outputs]
        chosen = choose_best_sample(samples, finish_reasons)
        chosen.update({"id": item["id"], "phase_used": 3})
        response_records[str(item["id"])] = chosen
        write_checkpoint(CHECKPOINT_PATH, response_records)

missing_ids = [item["id"] for item in data_run if str(item["id"]) not in response_records]
if missing_ids:
    raise RuntimeError(f"Missing responses for ids: {missing_ids[:10]} ... total={len(missing_ids)}")

responses = [response_records[str(item["id"])] ["response"] for item in data_run]
phase_counts = Counter(response_records[str(item["id"])].get("phase_used") for item in data_run)
uncertain_count = sum(bool(response_records[str(item["id"])].get("uncertain")) for item in data_run)
finish_counts = Counter(str(response_records[str(item["id"])].get("finish_reason")) for item in data_run)

print("Adaptive generation complete.")
print("len(data_run):", len(data_run))
print("len(responses):", len(responses))
print("Phase counts:", dict(sorted(phase_counts.items())))
print("Still marked uncertain:", uncertain_count)
print("Finish reasons:", dict(finish_counts))
print("Checkpoint:", CHECKPOINT_PATH)

for i in range(min(3, len(responses))):
    rec = response_records[str(data_run[i]["id"])]
    print(f"\nResponse {i} (id={data_run[i].get('id')}, phase={rec.get('phase_used')})")
    print(responses[i][:600], "..." if len(responses[i]) > 600 else "")


### Generate with Transformers (for Datahub)

In [9]:
# --- Transformers generation path (disabled when using vLLM above — do not run both) ---
# prompts = []
# for item in data_run:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#         enable_thinking=True,
#     )
#     prompts.append(prompt_text)
#
# responses = []
# print(
#     f"Generating {len(prompts)} response(s), max_new_tokens={MAX_TOKENS} each "
#     "(raise MAX_TOKENS in config if answers get cut off).",
#     flush=True,
# )
#
# with torch.no_grad():
#     for prompt_text in tqdm(prompts, desc="Generate", unit="q"):
#         inputs = tokenizer(
#             prompt_text,
#             return_tensors="pt",
#             truncation=True,
#             max_length=16384,
#         ).to(_device)
#         output_ids = llm.generate(
#             **inputs,
#             max_new_tokens=MAX_TOKENS,
#             temperature=0.6,
#             top_p=0.95,
#             top_k=20,
#             min_p=0.0,
#             repetition_penalty=1.0,
#             do_sample=True,
#             pad_token_id=tokenizer.eos_token_id,
#         )
#         new_tokens = output_ids[0, inputs["input_ids"].shape[1] :]
#         responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
#
# print(f"Done. len(responses)={len(responses)} (expected {len(data_run)}).", flush=True)
#
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data_run[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")


## 7. Score Responses

**Requires** `answer` in each row (public set). If you loaded `private.jsonl`, skip this section and only export responses.

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [10]:
print("N_QUESTIONS:", N_QUESTIONS)
print("len(data_run):", len(data_run))
print("len(responses):", len(responses) if "responses" in globals() else "missing")
print("len(results):", len(results) if "results" in globals() else "missing")
print([x["id"] for x in data_run[:10]])

N_QUESTIONS: 50
len(data_run): 50
len(responses): 50
len(results): missing
[228, 51, 563, 501, 457, 285, 209, 1116, 178, 864]


In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


from judger import Judger

if "responses" not in globals():
    raise NameError(
        "`responses` is not defined. Finish the **Generate** cell above first. "
        "If you interrupted it before, that variable is never assigned — re-run Generate to completion."
    )
if len(responses) != len(data_run):
    raise ValueError(
        f"len(responses)={len(responses)} but len(data_run)={len(data_run)}. "
        "Re-run Generate for all items, or shrink data_run / N_QUESTIONS and regenerate."
    )

judger = Judger(strict_extract=False)

if not all("answer" in item for item in data_run):
    raise ValueError(
        "Scoring needs an `answer` field per row. Use `public.jsonl` for section 7, "
        "or skip scoring for the private set."
    )

print(
    f"Scoring {len(data_run)} responses — MCQ is instant; free-form `auto_judge` can take minutes "
    "on the first few items. You should see the line below update each question.",
    flush=True,
)

results = []
with tqdm(total=len(data_run), desc="Scoring", unit="q", dynamic_ncols=True) as pbar:
    for item, response in zip(data_run, responses):
        pbar.set_postfix_str(f"id={item.get('id')}", refresh=True)

        is_mcq = bool(item.get("options"))
        gold = item["answer"]

        if is_mcq:
            correct = score_mcq(response, str(gold))
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            try:
                correct = judger.auto_judge(
                    pred=response,
                    gold=gold_list,
                    options=[[]] * len(gold_list),
                )
            except Exception:
                correct = False

        results.append(
            {
                "id": item.get("id"),
                "is_mcq": is_mcq,
                "gold": gold,
                "response": response,
                "correct": correct,
            }
        )
        pbar.update(1)

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = DATA_MODE == "public"  # Private has no labels; set False automatically.

if "responses" not in globals():
    raise RuntimeError("Run Section 6 generation before saving results.")
if len(responses) != len(data_run):
    raise RuntimeError(f"responses length {len(responses)} != data_run length {len(data_run)}")

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

if SAVE_EVAL:
    if "results" not in globals():
        raise RuntimeError("SAVE_EVAL=True requires running Section 7 scoring first.")
    records = []
    for r in results:
        meta = response_records.get(str(r["id"]), {}) if "response_records" in globals() else {}
        records.append({
            "id": r["id"],
            "is_mcq": r["is_mcq"],
            "gold": r["gold"],
            "response": r["response"],
            "correct": r["correct"],
            "phase_used": meta.get("phase_used"),
            "uncertain": meta.get("uncertain"),
            "finish_reason": meta.get("finish_reason"),
            "consensus_count": meta.get("consensus_count"),
            "n_samples": meta.get("n_samples"),
        })
else:
    records = []
    for item, response in zip(data_run, responses):
        meta = response_records.get(str(item["id"]), {}) if "response_records" in globals() else {}
        records.append({
            "id": item["id"],
            "is_mcq": bool(item.get("options")),
            "response": response,
            "phase_used": meta.get("phase_used"),
            "uncertain": meta.get("uncertain"),
            "finish_reason": meta.get("finish_reason"),
            "consensus_count": meta.get("consensus_count"),
            "n_samples": meta.get("n_samples"),
        })

with open(out_path, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(records)} records to {out_path}")


## 10. Generate Submission CSV

Convert the saved JSONL into a competition-format CSV (`id,response`).
Works for both public (evaluation) runs and private (submission) runs.
The `response` column holds the **full model trace** (competition requirement).

In [ ]:
import csv
from datetime import date

# ── Submission CSV generation ─────────────────────────────────────────────────
# Reads the JSONL saved in Section 9 and writes a properly quoted CSV.
# Set SUBMISSION_INPUT to the path you just saved.
SUBMISSION_INPUT = OUTPUT_PATH
SUBMISSION_OUTPUT = (
    REPO_ROOT / "artifacts" / "submissions"
    / f"submission_{date.today().isoformat()}.csv"
)

SUBMISSION_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

rows = []
with open(SUBMISSION_INPUT, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        rows.append({"id": rec["id"], "response": rec["response"]})

rows.sort(key=lambda r: r["id"])

with open(SUBMISSION_OUTPUT, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "response"], quoting=csv.QUOTE_ALL)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} rows -> {SUBMISSION_OUTPUT}")
print("First 3 rows preview:")
for r in rows[:3]:
    print(f"  id={r['id']}  response={r['response'][:80]}...")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!